In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=os.getenv("OPENAI_BASE_URL"), api_key=os.getenv("OPENAI_API_KEY"))

# [Legacy] 기존 방식: {"type": "retrieval"}
# 최신 방식: {"type": "file_search"}
# assistant = client.beta.assistants.create(
#     name="Book Assistant",
#     instructions="You help users with their question on the files they upload.",
#     model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
#     tools=[{"type": "file_search"}],
# )
assistant_id = "asst_8g39VCPq4Zcq8MxvNh9SepF6"

In [ ]:
thread = client.beta.threads.create(
    messages=[
        {
            "role": "user",
            "content": "I want you to help me with this file",
        }
    ]
)
thread

In [ ]:
file = client.files.create(
    file=open("./files/chapter_one.txt", "rb"), purpose="assistants"
)
file

In [ ]:
# [Legacy] 기존 방식: file_ids=[file.id]
# 최신 방식: attachments=[{"file_id": file.id, "tools": [{"type": "file_search"}]}]
client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="",
    attachments=[{"file_id": file.id, "tools": [{"type": "file_search"}]}],
)

In [ ]:
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant_id,
)
run

In [ ]:
import json


def get_run(run_id, thread_id):
    return client.beta.threads.runs.retrieve(
        run_id=run_id,
        thread_id=thread_id,
    )


def send_message(thread_id, content):
    return client.beta.threads.messages.create(
        thread_id=thread_id, role="user", content=content
    )


def get_messages(thread_id):
    messages = client.beta.threads.messages.list(thread_id=thread_id)
    messages = list(messages)
    messages.reverse()
    for message in messages:
        print(f"{message.role}: {message.content[0].text.value}")
        for annotation in message.content[0].text.annotations:
            print(f"Source: {annotation.file_citation}")


def get_tool_outputs(run_id, thread_id):
    run = get_run(run_id, thread_id)
    outputs = []
    for action in run.required_action.submit_tool_outputs.tool_calls:
        action_id = action.id
        function = action.function
        print(f"Calling function: {function.name} with arg {function.arguments}")
        outputs.append(
            {
                "output": functions_map[function.name](json.loads(function.arguments)),
                "tool_call_id": action_id,
            }
        )
    return outputs


def submit_tool_outputs(run_id, thread_id):
    outpus = get_tool_outputs(run_id, thread_id)
    return client.beta.threads.runs.submit_tool_outputs(
        run_id=run_id,
        thread_id=thread_id,
        tool_outputs=outpus,
    )

In [ ]:
get_run(run.id, thread.id).status

In [ ]:
get_messages(thread.id)

In [ ]:
send_message(
    thread.id,
    "Where does he work?",
)